# LightGBM-Huber on R1000/R2000 Daily Holdings

## Goal

This standalone notebook trains a robust LightGBM return regressor on
`manager_holdings/R1000_R2000_daily_turnover21D.parquet`. It builds the original
cross-sectional feature vector:

\[
[r_{1d},r_{5d},r_{21d},r_{63d},r_{126d},r_{252d},
\sigma_{21d},\sigma_{63d},turnover_{21d},\log(price),\log(market\ cap)].
\]

Run it from the directory containing `manager_holdings/`. If dependencies are
missing, run once and restart the kernel:

```python
%pip install -q "lightgbm>=4.0" "scikit-learn>=1.3" "pandas>=2.0" "pyarrow>=12" "numpy>=1.24" "matplotlib>=3.7"
```

No volume, turnover, price, or size screen is applied. Zero-volume and
low-volume securities remain in the universe.

## Context & Methods

### Key assumptions

The key configuration is `TRET_T1D_ROLE`:

- `"past"`: raw `TRET_T1D` is the return ending at `day=t`; it enters the
  historical feature stream, and the next global market date supplies the label.
- `"future"`: raw `TRET_T1D` is already the `t` to `t+1` label; the value from
  the previous global market date supplies the historical one-day return.

The notebook never uses the next available observation when a security is absent
on the adjacent global market date. Multi-day returns are compounded;
volatilities use `ddof=0`. `RETURN_MULTIPLIER=1.0` assumes decimal returns.

Price is auto-detected from `price`, `PRICE`, `prc`, or `PRC`. If price is absent,
the notebook explicitly runs ten available features unless `REQUIRE_PRICE=True`.

## Setup

In [ ]:
from __future__ import annotations

import json
import math
import os
import platform
from pathlib import Path
import warnings

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


def env_bool(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "y", "on"}:
        return True
    if normalized in {"0", "false", "no", "n", "off"}:
        return False
    raise ValueError(f"{name} must be boolean-like, got {value!r}.")


def optional_timestamp(name: str, default: str | None) -> pd.Timestamp | None:
    value = os.environ.get(name, default)
    if value is None or not str(value).strip():
        return None
    return pd.Timestamp(value)


DATA_PATH = Path(os.environ.get(
    "HOLDINGS_DATA_PATH",
    "manager_holdings/R1000_R2000_daily_turnover21D.parquet",
)).expanduser()
OUTPUT_DIR = Path(os.environ.get(
    "HOLDINGS_OUTPUT_DIR", "lightgbm_huber_R1000_R2000_results"
)).expanduser()

# IMPORTANT: set to "past" or "future" after checking the data dictionary.
TRET_T1D_ROLE = os.environ.get("TRET_T1D_ROLE", "past").strip().lower()
PRICE_COLUMN = os.environ.get("PRICE_COLUMN", "auto").strip()
REQUIRE_PRICE = env_bool("REQUIRE_PRICE", False)

TRAIN_END = optional_timestamp("TRAIN_END", "2022-12-31")
VALIDATION_START = optional_timestamp("VALIDATION_START", "2023-01-01")
VALIDATION_END = optional_timestamp("VALIDATION_END", "2023-12-31")
TEST_START = optional_timestamp("TEST_START", "2024-01-01")
TEST_END = optional_timestamp("TEST_END", None)

RETURN_MULTIPLIER = float(os.environ.get("RETURN_MULTIPLIER", "1.0"))
TOP_K = int(os.environ.get("TOP_K", "10"))
SEED = int(os.environ.get("SEED", "1337"))
N_ESTIMATORS = int(os.environ.get("LGBM_N_ESTIMATORS", "500"))
EARLY_STOPPING_ROUNDS = int(os.environ.get("LGBM_EARLY_STOPPING_ROUNDS", "30"))
N_JOBS = int(os.environ.get("LGBM_N_JOBS", "-1"))
RUN_ABLATIONS = env_bool("RUN_ABLATIONS", True)
SAVE_MODEL = env_bool("SAVE_MODEL", False)
SAVE_ROW_LEVEL_PREDICTIONS = env_bool("SAVE_ROW_LEVEL_PREDICTIONS", False)

if TRET_T1D_ROLE not in {"past", "future"}:
    raise ValueError("TRET_T1D_ROLE must be exactly 'past' or 'future'.")
if not (TRAIN_END < VALIDATION_START <= VALIDATION_END < TEST_START):
    raise ValueError("Train, validation, and test dates must be chronological and disjoint.")
if TEST_END is not None and TEST_START > TEST_END:
    raise ValueError("TEST_END must be on or after TEST_START.")
if min(TOP_K, N_ESTIMATORS, EARLY_STOPPING_ROUNDS) < 1:
    raise ValueError("TOP_K, N_ESTIMATORS, and EARLY_STOPPING_ROUNDS must be positive.")
if not math.isfinite(RETURN_MULTIPLIER) or RETURN_MULTIPLIER == 0:
    raise ValueError("RETURN_MULTIPLIER must be finite and nonzero.")

display(pd.Series({
    "data_path": str(DATA_PATH),
    "output_dir": str(OUTPUT_DIR),
    "TRET_T1D_role": TRET_T1D_ROLE,
    "price_column_request": PRICE_COLUMN,
    "require_price": REQUIRE_PRICE,
    "train_end": TRAIN_END.date().isoformat(),
    "validation": f"{VALIDATION_START.date()} to {VALIDATION_END.date()}",
    "test": f"{TEST_START.date()} to {TEST_END.date() if TEST_END is not None else 'latest'}",
    "return_multiplier": RETURN_MULTIPLIER,
    "top_k": TOP_K,
    "run_ablations": RUN_ABLATIONS,
}, name="value").to_frame())

## Data

### 1. Load and validate the input

The six named fields are required. Price is optional and schema-detected before
loading. Missing features stay in the sample for LightGBM-native handling; only
missing/non-finite aligned targets are excluded from supervised estimation.

In [ ]:
BASE_REQUIRED_COLUMNS = [
    "DOLLARVOLUME_AVG21D", "day", "security", "TRET_T1D",
    "market_cap", "turnover_21day",
]

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Input not found: {DATA_PATH.resolve()}\n"
        "Place the notebook beside manager_holdings/, or set HOLDINGS_DATA_PATH."
    )

schema_columns = list(pq.ParquetFile(DATA_PATH).schema.names)
missing = sorted(set(BASE_REQUIRED_COLUMNS).difference(schema_columns))
if missing:
    raise ValueError(f"Parquet is missing required columns: {missing}")

if PRICE_COLUMN.lower() == "auto":
    DETECTED_PRICE_COLUMN = next(
        (name for name in ("price", "PRICE", "prc", "PRC") if name in schema_columns),
        None,
    )
else:
    DETECTED_PRICE_COLUMN = PRICE_COLUMN if PRICE_COLUMN in schema_columns else None
    if DETECTED_PRICE_COLUMN is None and REQUIRE_PRICE:
        raise ValueError(f"Configured PRICE_COLUMN={PRICE_COLUMN!r} is absent.")

if DETECTED_PRICE_COLUMN is None:
    message = "No price column found; running the explicit ten-feature model without log_price."
    if REQUIRE_PRICE:
        raise ValueError(message)
    warnings.warn(message)

load_columns = BASE_REQUIRED_COLUMNS + (
    [DETECTED_PRICE_COLUMN] if DETECTED_PRICE_COLUMN is not None else []
)
frame = pd.read_parquet(DATA_PATH, columns=load_columns).copy()
frame["day"] = pd.to_datetime(frame["day"], errors="coerce").dt.normalize()
if frame["day"].isna().any():
    raise ValueError(f"day contains {int(frame['day'].isna().sum()):,} unparseable values.")
if frame["security"].isna().any():
    raise ValueError(f"security contains {int(frame['security'].isna().sum()):,} missing values.")
frame["security"] = frame["security"].astype("string")
if frame.duplicated(["day", "security"]).any():
    raise ValueError("Expected at most one row per (day, security).")

numeric_columns = ["DOLLARVOLUME_AVG21D", "TRET_T1D", "market_cap", "turnover_21day"]
if DETECTED_PRICE_COLUMN is not None:
    numeric_columns.append(DETECTED_PRICE_COLUMN)
for column in numeric_columns:
    frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame.loc[~np.isfinite(frame[column]), column] = np.nan
frame["TRET_T1D"] *= RETURN_MULTIPLIER

invalid_returns = frame["TRET_T1D"] < -1.0
if invalid_returns.any():
    warnings.warn(f"Treating {int(invalid_returns.sum()):,} returns below -100% as missing.")
    frame.loc[invalid_returns, "TRET_T1D"] = np.nan
for column in ("DOLLARVOLUME_AVG21D", "turnover_21day"):
    invalid = frame[column] < 0
    if invalid.any():
        warnings.warn(f"Treating {int(invalid.sum()):,} negative {column} values as missing.")
        frame.loc[invalid, column] = np.nan
frame.loc[frame["market_cap"] <= 0, "market_cap"] = np.nan
frame["_price_raw"] = (
    frame[DETECTED_PRICE_COLUMN].abs().where(frame[DETECTED_PRICE_COLUMN].abs() > 0)
    if DETECTED_PRICE_COLUMN is not None else np.nan
)
frame = frame.sort_values(["day", "security"], kind="stable").reset_index(drop=True)

target_abs = frame["TRET_T1D"].abs().dropna()
display(pd.Series({
    "rows": len(frame),
    "market_dates": frame["day"].nunique(),
    "securities": frame["security"].nunique(),
    "first_day": frame["day"].min().date().isoformat(),
    "last_day": frame["day"].max().date().isoformat(),
    "detected_price_column": DETECTED_PRICE_COLUMN,
    "missing_raw_return_rows": int(frame["TRET_T1D"].isna().sum()),
    "zero_dollar_volume_rows_kept": int(frame["DOLLARVOLUME_AVG21D"].eq(0).sum()),
    "zero_turnover_rows_kept": int(frame["turnover_21day"].eq(0).sum()),
    "median_abs_raw_return": float(target_abs.median()) if len(target_abs) else np.nan,
    "raw_return_abs_p99": float(target_abs.quantile(0.99)) if len(target_abs) else np.nan,
}, name="value").to_frame())
display(frame.head(5))
if target_abs.empty:
    raise ValueError("TRET_T1D has no finite observations.")
if target_abs.median() > 0.20:
    warnings.warn("Median absolute TRET_T1D exceeds 20%; verify RETURN_MULTIPLIER.")

### 2. Align returns, construct rolling features, and split by date

In [ ]:
market_dates = pd.Index(frame["day"].drop_duplicates().sort_values())
market_date_number = pd.Series(np.arange(len(market_dates)), index=market_dates)
frame["_market_date_number"] = frame["day"].map(market_date_number).astype(int)

raw_return = frame["TRET_T1D"].to_numpy(dtype=float)
historical_return = np.full(len(frame), np.nan, dtype=float)
target_return = np.full(len(frame), np.nan, dtype=float)

for positions in frame.groupby("security", sort=False).indices.values():
    positions = np.asarray(positions, dtype=int)
    dates = frame.loc[positions, "_market_date_number"].to_numpy(dtype=int)
    values = raw_return[positions]
    adjacent = np.diff(dates) == 1
    left = positions[:-1][adjacent]
    right = positions[1:][adjacent]
    if TRET_T1D_ROLE == "past":
        historical_return[positions] = values
        target_return[left] = values[1:][adjacent]
    else:
        target_return[positions] = values
        historical_return[right] = values[:-1][adjacent]

frame["return_1d"] = historical_return
frame["target_return"] = target_return


def rolling_compounded_return(values: np.ndarray, dates: np.ndarray, window: int) -> np.ndarray:
    output = np.full(len(values), np.nan, dtype=float)
    if len(values) < window:
        return output
    factors = 1.0 + values
    valid = np.isfinite(values) & (factors >= 0.0)
    positive = valid & (factors > 0.0)
    zeros = valid & (factors == 0.0)
    logs = np.zeros(len(values), dtype=float)
    logs[positive] = np.log(factors[positive])
    invalid_prefix = np.r_[0, np.cumsum(~valid)]
    zero_prefix = np.r_[0, np.cumsum(zeros)]
    log_prefix = np.r_[0.0, np.cumsum(logs)]
    end = np.arange(window - 1, len(values))
    start = end - window + 1
    usable = (
        (invalid_prefix[end + 1] - invalid_prefix[start] == 0)
        & (dates[end] - dates[start] == window - 1)
    )
    zero_count = zero_prefix[end + 1] - zero_prefix[start]
    log_sum = log_prefix[end + 1] - log_prefix[start]
    output[end[usable & (zero_count > 0)]] = -1.0
    normal = usable & (zero_count == 0)
    with np.errstate(over="ignore", invalid="ignore"):
        output[end[normal]] = np.expm1(log_sum[normal])
    return output


def rolling_population_volatility(values: np.ndarray, dates: np.ndarray, window: int) -> np.ndarray:
    output = np.full(len(values), np.nan, dtype=float)
    if len(values) < window:
        return output
    valid = np.isfinite(values)
    safe = np.where(valid, values, 0.0)
    count_prefix = np.r_[0, np.cumsum(valid)]
    sum_prefix = np.r_[0.0, np.cumsum(safe)]
    square_prefix = np.r_[0.0, np.cumsum(safe * safe)]
    end = np.arange(window - 1, len(values))
    start = end - window + 1
    usable = (
        (count_prefix[end + 1] - count_prefix[start] == window)
        & (dates[end] - dates[start] == window - 1)
    )
    total = sum_prefix[end + 1] - sum_prefix[start]
    total_square = square_prefix[end + 1] - square_prefix[start]
    variance = np.maximum(total_square / window - (total / window) ** 2, 0.0)
    output[end[usable]] = np.sqrt(variance[usable])
    return output


RETURN_WINDOWS = (1, 5, 21, 63, 126, 252)
VOLATILITY_WINDOWS = (21, 63)
for window in RETURN_WINDOWS[1:]:
    frame[f"return_{window}d"] = np.nan
for window in VOLATILITY_WINDOWS:
    frame[f"volatility_{window}d"] = np.nan

for positions in frame.groupby("security", sort=False).indices.values():
    positions = np.asarray(positions, dtype=int)
    values = frame.loc[positions, "return_1d"].to_numpy(dtype=float)
    dates = frame.loc[positions, "_market_date_number"].to_numpy(dtype=int)
    for window in RETURN_WINDOWS[1:]:
        frame.loc[positions, f"return_{window}d"] = rolling_compounded_return(values, dates, window)
    for window in VOLATILITY_WINDOWS:
        frame.loc[positions, f"volatility_{window}d"] = rolling_population_volatility(values, dates, window)

frame["log_dollar_volume_21d"] = np.log1p(frame["DOLLARVOLUME_AVG21D"])
frame["log_market_cap"] = np.log(frame["market_cap"])
frame["log_price"] = np.log(frame["_price_raw"])

RETURN_VOLATILITY_FEATURES = [
    "return_1d", "return_5d", "return_21d", "return_63d", "return_126d", "return_252d",
    "volatility_21d", "volatility_63d",
]
PRIMARY_FEATURES = RETURN_VOLATILITY_FEATURES + ["turnover_21day"]
if DETECTED_PRICE_COLUMN is not None:
    PRIMARY_FEATURES += ["log_price"]
PRIMARY_FEATURES += ["log_market_cap"]
PRIMARY_MODEL_NAME = "all_eleven" if DETECTED_PRICE_COLUMN is not None else "all_available_10"

FEATURE_SETS = {PRIMARY_MODEL_NAME: PRIMARY_FEATURES}
if RUN_ABLATIONS:
    FEATURE_SETS.update({
        "returns_volatility_only": RETURN_VOLATILITY_FEATURES,
        "rv_plus_turnover": RETURN_VOLATILITY_FEATURES + ["turnover_21day"],
        "rv_plus_size": RETURN_VOLATILITY_FEATURES + ["log_market_cap"],
        "liquidity_size_only": ["log_dollar_volume_21d", "log_market_cap", "turnover_21day"],
        "turnover_only": ["turnover_21day"],
    })
    if DETECTED_PRICE_COLUMN is not None:
        FEATURE_SETS["rv_plus_price"] = RETURN_VOLATILITY_FEATURES + ["log_price"]

audit_security = frame.groupby("security", sort=False).size().idxmax()
alignment_audit = frame.loc[
    frame["security"] == audit_security,
    ["day", "security", "TRET_T1D", "return_1d", "target_return"],
].head(10)
print(f"Alignment audit for TRET_T1D_ROLE={TRET_T1D_ROLE!r}")
display(alignment_audit)

finite_target = frame["target_return"].notna()
train_mask = finite_target & (frame["day"] <= TRAIN_END)
validation_mask = finite_target & frame["day"].between(VALIDATION_START, VALIDATION_END)
test_mask = finite_target & (frame["day"] >= TEST_START)
if TEST_END is not None:
    test_mask &= frame["day"] <= TEST_END
train, validation, test = (frame.loc[mask].copy() for mask in (train_mask, validation_mask, test_mask))

for name, split in {"train": train, "validation": validation, "test": test}.items():
    if split.empty or split["day"].nunique() < 2:
        raise ValueError(f"{name} split needs at least two labeled market dates.")
minimum_test_assets = int(test.groupby("day", sort=True).size().min())
if minimum_test_assets < TOP_K:
    raise ValueError(f"A test date has only {minimum_test_assets} assets, fewer than TOP_K={TOP_K}.")

split_table = pd.DataFrame([
    {
        "split": name,
        "start": split["day"].min().date().isoformat(),
        "end": split["day"].max().date().isoformat(),
        "dates": split["day"].nunique(),
        "rows": len(split),
        "securities": split["security"].nunique(),
        "minimum_daily_assets": split.groupby("day").size().min(),
    }
    for name, split in (("train", train), ("validation", validation), ("test", test))
])
display(split_table)
display(pd.Series({name: " | ".join(columns) for name, columns in FEATURE_SETS.items()}, name="features").to_frame())

## Results

### 3. Define the model and evaluation functions

In [ ]:
def fit_huber(feature_columns: list[str]) -> lgb.LGBMRegressor:
    min_child_samples = min(200, max(1, len(train) // 100))
    model = lgb.LGBMRegressor(
        objective="huber",
        alpha=0.9,
        n_estimators=N_ESTIMATORS,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=min_child_samples,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        max_bin=127,
        random_state=SEED,
        n_jobs=N_JOBS,
        verbosity=-1,
        deterministic=True,
        force_col_wise=True,
    )
    model.fit(
        train[feature_columns].astype("float32"),
        train["target_return"].astype("float32"),
        eval_set=[
            (
                validation[feature_columns].astype("float32"),
                validation["target_return"].astype("float32"),
            )
        ],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
            lgb.log_evaluation(period=50),
        ],
    )
    return model


def hac_mean_t_stat(values: pd.Series, max_lag: int = 5) -> float:
    # Newey-West t-statistic for a daily mean with Bartlett weights.
    array = pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)
    array = array[np.isfinite(array)]
    count = len(array)
    if count < 2:
        return math.nan
    centered = array - array.mean()
    long_run_variance = float(centered @ centered / count)
    for lag in range(1, min(max_lag, count - 1) + 1):
        autocovariance = float(centered[lag:] @ centered[:-lag] / count)
        weight = 1.0 - lag / (max_lag + 1.0)
        long_run_variance += 2.0 * weight * autocovariance
    if not math.isfinite(long_run_variance) or long_run_variance <= 0:
        return math.nan
    standard_error = math.sqrt(long_run_variance / count)
    return float(array.mean() / standard_error)


def max_drawdown(returns: pd.Series) -> float:
    daily = pd.to_numeric(returns, errors="coerce")
    daily = daily[np.isfinite(daily)]
    if daily.empty or (daily < -1.0).any():
        return math.nan
    equity = (1.0 + daily).cumprod()
    running_peak = np.maximum.accumulate(np.r_[1.0, equity.to_numpy()])[1:]
    return float(np.min(equity.to_numpy() / running_peak - 1.0))


def return_summary(values: pd.Series) -> dict[str, float | int]:
    daily = pd.to_numeric(values, errors="coerce")
    daily = daily[np.isfinite(daily)]
    observations = len(daily)
    if observations == 0:
        return {
            "observations": 0,
            "annualized_mean": math.nan,
            "annualized_volatility": math.nan,
            "sharpe": math.nan,
            "max_drawdown": math.nan,
        }
    annualized_mean = float(daily.mean() * 252.0)
    annualized_volatility = (
        float(daily.std(ddof=1) * math.sqrt(252.0))
        if observations > 1
        else math.nan
    )
    sharpe = (
        annualized_mean / annualized_volatility
        if annualized_volatility > 0 and math.isfinite(annualized_volatility)
        else math.nan
    )
    return {
        "observations": observations,
        "annualized_mean": annualized_mean,
        "annualized_volatility": annualized_volatility,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown(daily),
    }


def score_test(model: lgb.LGBMRegressor, feature_columns: list[str]) -> pd.DataFrame:
    prediction_columns = list(dict.fromkeys([
        "day", "security", "TRET_T1D", "return_1d", "target_return",
        "DOLLARVOLUME_AVG21D", "market_cap", "turnover_21day",
        *feature_columns,
    ]))
    predictions = test[prediction_columns].copy()
    best_iteration = int(model.best_iteration_ or model.n_estimators)
    predictions["score"] = model.predict(
        test[feature_columns].astype("float32"), num_iteration=best_iteration
    )
    if not np.isfinite(predictions["score"]).all():
        raise ValueError("LightGBM produced a non-finite test score.")
    return predictions


def daily_rank_and_portfolios(predictions: pd.DataFrame) -> pd.DataFrame:
    records: list[dict[str, float | int | pd.Timestamp]] = []
    for day, group in predictions.groupby("day", sort=True):
        ranked = group.sort_values(
            ["score", "security"], ascending=[False, True], kind="stable"
        )
        limit = min(TOP_K, len(ranked))
        top = ranked.head(limit)
        bottom = ranked.tail(limit)
        score_rank = ranked["score"].rank(method="average")
        return_rank = ranked["target_return"].rank(method="average")
        rank_ic = score_rank.corr(return_rank)
        top_return = float(top["target_return"].mean())
        bottom_return = float(bottom["target_return"].mean())
        records.append(
            {
                "day": day,
                "rank_ic": float(rank_ic) if pd.notna(rank_ic) else math.nan,
                "top_return": top_return,
                "bottom_return": bottom_return,
                "spread_return": top_return - bottom_return,
                "top_holdings": len(top),
                "bottom_holdings": len(bottom),
                "eligible_assets": len(ranked),
            }
        )
    return pd.DataFrame.from_records(records)


def model_summary(
    model_name: str,
    feature_columns: list[str],
    model: lgb.LGBMRegressor,
    daily: pd.DataFrame,
) -> dict[str, float | int | str]:
    top = return_summary(daily["top_return"])
    bottom = return_summary(daily["bottom_return"])
    spread = return_summary(daily["spread_return"])
    return {
        "model": model_name,
        "features": " | ".join(feature_columns),
        "best_iteration": int(model.best_iteration_ or model.n_estimators),
        "train_rows": len(train),
        "validation_rows": len(validation),
        "test_rows": len(test),
        "test_dates": int(daily["day"].nunique()),
        "rank_ic": float(daily["rank_ic"].mean()),
        "top_annualized_mean": top["annualized_mean"],
        "top_sharpe": top["sharpe"],
        "bottom_annualized_mean": bottom["annualized_mean"],
        "bottom_sharpe": bottom["sharpe"],
        "spread_annualized_mean": spread["annualized_mean"],
        "spread_annualized_volatility": spread["annualized_volatility"],
        "spread_sharpe": spread["sharpe"],
        "spread_max_drawdown": spread["max_drawdown"],
        "spread_hac5_t": hac_mean_t_stat(daily["spread_return"], max_lag=5),
    }


def annual_rows(model_name: str, daily: pd.DataFrame) -> list[dict[str, float | int | str]]:
    rows: list[dict[str, float | int | str]] = []
    for year, group in daily.groupby(daily["day"].dt.year, sort=True):
        top = return_summary(group["top_return"])
        bottom = return_summary(group["bottom_return"])
        spread = return_summary(group["spread_return"])
        rows.append(
            {
                "model": model_name,
                "period": str(int(year)),
                "start": group["day"].min().date().isoformat(),
                "end": group["day"].max().date().isoformat(),
                "observations": len(group),
                "rank_ic": float(group["rank_ic"].mean()),
                "top_annualized_mean": top["annualized_mean"],
                "top_sharpe": top["sharpe"],
                "bottom_annualized_mean": bottom["annualized_mean"],
                "bottom_sharpe": bottom["sharpe"],
                "spread_annualized_mean": spread["annualized_mean"],
                "spread_sharpe": spread["sharpe"],
                "spread_hac5_t": hac_mean_t_stat(group["spread_return"], max_lag=5),
            }
        )
    return rows


def selection_characteristic_rows(
    model_name: str, predictions: pd.DataFrame
) -> list[dict[str, float | int | str]]:
    daily_rows: list[dict[str, float | int | str]] = []
    for day, group in predictions.groupby("day", sort=True):
        ranked = group.sort_values(
            ["score", "security"], ascending=[False, True], kind="stable"
        )
        limit = min(TOP_K, len(ranked))
        selections = {
            "top": ranked.head(limit),
            "universe": ranked,
            "bottom": ranked.tail(limit),
        }
        for leg, selected in selections.items():
            daily_rows.append(
                {
                    "model": model_name,
                    "day": day,
                    "leg": leg,
                    "holdings": len(selected),
                    "median_dollar_volume_21d": float(selected["DOLLARVOLUME_AVG21D"].median()),
                    "median_market_cap": float(selected["market_cap"].median()),
                    "median_turnover_21day": float(selected["turnover_21day"].median()),
                    "zero_dollar_volume_share": float(selected["DOLLARVOLUME_AVG21D"].eq(0).mean()),
                }
            )
    daily_frame = pd.DataFrame.from_records(daily_rows)
    result = (
        daily_frame.groupby(["model", "leg"], sort=True, as_index=False)
        .agg(
            dates=("day", "nunique"),
            mean_holdings=("holdings", "mean"),
            mean_daily_median_dollar_volume_21d=("median_dollar_volume_21d", "mean"),
            mean_daily_median_market_cap=("median_market_cap", "mean"),
            mean_daily_median_turnover_21day=("median_turnover_21day", "mean"),
            mean_zero_dollar_volume_share=("zero_dollar_volume_share", "mean"),
        )
    )
    return result.to_dict(orient="records")

### 4. Train the primary model and optional feature ablations

In [ ]:
models: dict[str, lgb.LGBMRegressor] = {}
predictions_by_model: dict[str, pd.DataFrame] = {}
daily_by_model: dict[str, pd.DataFrame] = {}
summary_rows: list[dict[str, float | int | str]] = []
year_rows: list[dict[str, float | int | str]] = []
importance_rows: list[dict[str, float | int | str]] = []
characteristic_rows: list[dict[str, float | int | str]] = []

for model_name, feature_columns in FEATURE_SETS.items():
    print(f"Training {model_name}: {feature_columns}")
    fitted = fit_huber(feature_columns)
    scored = score_test(fitted, feature_columns)
    daily = daily_rank_and_portfolios(scored)
    models[model_name] = fitted
    predictions_by_model[model_name] = scored
    daily_by_model[model_name] = daily
    summary_rows.append(model_summary(model_name, feature_columns, fitted, daily))
    year_rows.extend(annual_rows(model_name, daily))
    characteristic_rows.extend(selection_characteristic_rows(model_name, scored))

    gain = fitted.booster_.feature_importance(importance_type="gain")
    gain_total = float(gain.sum())
    for feature, value in zip(feature_columns, gain):
        importance_rows.append(
            {
                "model": model_name,
                "feature": feature,
                "gain": float(value),
                "gain_share": float(value / gain_total) if gain_total > 0 else math.nan,
            }
        )

summary_table = pd.DataFrame.from_records(summary_rows)
year_table = pd.DataFrame.from_records(year_rows)
feature_importance = pd.DataFrame.from_records(importance_rows).sort_values(
    ["model", "gain"], ascending=[True, False], kind="stable"
)
selection_characteristics = pd.DataFrame.from_records(characteristic_rows)

display(summary_table.round(6))
display(year_table.round(6))
display(feature_importance.round(6))
display(selection_characteristics.round(6))

### 5. Save compact, reproducible outputs

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_table.to_csv(OUTPUT_DIR / "summary.csv", index=False)
year_table.to_csv(OUTPUT_DIR / "year_table.csv", index=False)
feature_importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
selection_characteristics.to_csv(
    OUTPUT_DIR / "selection_characteristics.csv", index=False
)

primary_daily = daily_by_model[PRIMARY_MODEL_NAME].copy()
primary_daily.to_parquet(OUTPUT_DIR / "daily_portfolios.parquet", index=False)

if SAVE_MODEL:
    for model_name, fitted in models.items():
        fitted.booster_.save_model(
            str(OUTPUT_DIR / f"{model_name}_model.txt"),
            num_iteration=int(fitted.best_iteration_ or fitted.n_estimators),
        )

if SAVE_ROW_LEVEL_PREDICTIONS:
    pd.concat(
        [values.assign(model=name) for name, values in predictions_by_model.items()],
        ignore_index=True,
    ).to_parquet(OUTPUT_DIR / "row_level_predictions.parquet", index=False)

run_config = {
    "data_path": str(DATA_PATH.resolve()),
    "data_rows": len(frame),
    "data_first_day": frame["day"].min().date().isoformat(),
    "data_last_day": frame["day"].max().date().isoformat(),
    "train_end": TRAIN_END.date().isoformat(),
    "validation_start": VALIDATION_START.date().isoformat(),
    "validation_end": VALIDATION_END.date().isoformat(),
    "test_start": TEST_START.date().isoformat(),
    "test_end": TEST_END.date().isoformat() if TEST_END is not None else None,
    "return_multiplier": RETURN_MULTIPLIER,
    "tret_t1d_role": TRET_T1D_ROLE,
    "detected_price_column": DETECTED_PRICE_COLUMN,
    "primary_model_name": PRIMARY_MODEL_NAME,
    "top_k": TOP_K,
    "seed": SEED,
    "n_estimators": N_ESTIMATORS,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    "run_ablations": RUN_ABLATIONS,
    "save_model": SAVE_MODEL,
    "save_row_level_predictions": SAVE_ROW_LEVEL_PREDICTIONS,
    "feature_sets": FEATURE_SETS,
    "zero_dollar_volume_rows_kept": int(frame["DOLLARVOLUME_AVG21D"].eq(0).sum()),
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "lightgbm_version": lgb.__version__,
}
(OUTPUT_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2, sort_keys=True), encoding="utf-8"
)

saved_files = pd.DataFrame(
    [
        {"file": path.name, "bytes": path.stat().st_size}
        for path in sorted(OUTPUT_DIR.iterdir())
        if path.is_file()
    ]
)
display(saved_files)
print(f"Saved compact results to: {OUTPUT_DIR.resolve()}")

### 6. Inspect Top, Bottom, and Top-minus-Bottom separately

In [ ]:
plot_frame = primary_daily.sort_values("day").copy()
plot_frame["cumulative_top_arithmetic"] = plot_frame["top_return"].cumsum()
plot_frame["cumulative_bottom_arithmetic"] = plot_frame["bottom_return"].cumsum()
plot_frame["cumulative_spread_arithmetic"] = plot_frame["spread_return"].cumsum()

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(
    plot_frame["day"],
    100 * plot_frame["cumulative_top_arithmetic"],
    label="Top-10",
    linewidth=1.8,
)
ax.plot(
    plot_frame["day"],
    100 * plot_frame["cumulative_bottom_arithmetic"],
    label="Bottom-10 underlying return",
    linewidth=1.8,
)
ax.plot(
    plot_frame["day"],
    100 * plot_frame["cumulative_spread_arithmetic"],
    label="Top minus Bottom",
    linewidth=2.3,
)
ax.axhline(0, color="black", linewidth=0.8, alpha=0.6)
ax.set_title("LightGBM-Huber: cumulative arithmetic daily returns")
ax.set_ylabel("Cumulative return (percentage points)")
ax.set_xlabel("Test date")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

## Takeaways

Read the executed output in this order:

1. Confirm the alignment audit matches the data dictionary. In `past` mode,
   raw return equals `return_1d`; in `future` mode, raw return equals the target.
2. Use RankIC for full-cross-section ordering, then inspect Top and Bottom
   separately to identify which leg creates the spread.
3. Compare the full model against returns/volatility, turnover, price, and size
   ablations. The output names a ten-feature model explicitly when price is absent.
4. Use HAC(5) for time-series evidence and selection characteristics to see
   whether the model moves toward smaller or less-liquid securities.

These are gross predictive results, not an execution-cost study.